# **02 Team Stats**


This notebook builds a league table comparing points to underlying xG, so you can see whether a team's results match how they're actually playing.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

TEAM = "Sunderland"
LEAGUE = "ENG-Premier League"
SEASON = "2025-2026"
DATA_FOLDER = Path("data")
CHART_FOLDER = Path("charts")
CHART_FOLDER.mkdir(parents=True, exist_ok=True)

print("Team:", TEAM)


Team: Sunderland


In [2]:
# load match  by match team stats

matches = pd.read_csv(DATA_FOLDER / "understat_team_stats.csv")
print("Rows, columns: ", matches.shape)
print("Columns: ", list(matches.columns))
matches.head()

Rows, columns:  (380, 29)
Columns:  ['league', 'season', 'game', 'league_id', 'season_id', 'game_id', 'date', 'home_team_id', 'away_team_id', 'home_team', 'away_team', 'away_team_code', 'home_team_code', 'home_points', 'home_expected_points', 'home_goals', 'home_xg', 'home_np_xg', 'home_np_xg_difference', 'home_ppda', 'home_deep_completions', 'away_points', 'away_expected_points', 'away_goals', 'away_xg', 'away_np_xg', 'away_np_xg_difference', 'away_ppda', 'away_deep_completions']


,league,season,game,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,...,home_ppda,home_deep_completions,away_points,away_expected_points,away_goals,away_xg,away_np_xg,away_np_xg_difference,away_ppda,away_deep_completions
0,ENG-Premier League,2526,2025-08-15 Liverpool-Bournemouth,1,2025,28778,2025-08-15 19:00:00,87,73,Liverpool,...,8.764706,9,0,0.8954,2,1.573030,1.573030,-0.757040,11.583333,8
1,ENG-Premier League,2526,2025-08-16 Aston Villa-Newcastle United,1,2025,28779,2025-08-16 11:30:00,71,86,Aston Villa,...,18.916667,2,1,2.3371,0,1.400980,1.400980,1.082379,6.083333,6
2,ENG-Premier League,2526,2025-08-16 Brighton-Fulham,1,2025,28780,2025-08-16 14:00:00,220,228,Brighton,...,8.928571,5,1,0.8350,1,0.902883,0.902883,0.223974,16.222222,6
3,ENG-Premier League,2526,2025-08-16 Sunderland-West Ham,1,2025,28781,2025-08-16 14:00:00,77,81,Sunderland,...,20.785714,2,0,1.1290,0,0.558948,0.558948,-0.165420,11.000000,8
4,ENG-Premier League,2526,2025-08-16 Tottenham-Burnley,1,2025,28782,2025-08-16 14:00:00,82,92,Tottenham,...,6.352941,7,0,0.4918,0,0.734138,0.734138,-1.244442,17.352941,6


Team xG is scattered across different columns, depending whether the team played home or away. This reshape gathers all the teams numbers into on consistent column.

In [11]:
# home table for every match
home_rows = pd.DataFrame({
    "date": matches["date"],
    "team": matches["home_team"],
    "opponent": matches["away_team"],
    "venue": "Home",
    "xg_for": matches["home_xg"],
    "xg_against": matches["away_xg"],
    "goals_for": matches["home_goals"],
    "goals_against": matches["away_goals"],
})

# away table for every match
away_rows = pd.DataFrame({
    "date": matches["date"],
    "team": matches["away_team"],
    "opponent": matches["home_team"],
    "venue": "Away",
    "xg_for": matches["away_xg"],
    "xg_against": matches["home_xg"],
    "goals_for": matches["away_goals"],
    "goals_against": matches["home_goals"],
})

# stack them into one table: two rows per match, one per team
team_matches_long = pd.concat([home_rows, away_rows], ignore_index=True)

# drop rows missing xg
team_matches_long = team_matches_long.dropna(subset=["xg_for", "goals_for"])

# sort alphabetically by team, then by date, so each team's matches are in order
team_matches_long = team_matches_long.sort_values(["team", "date"]).reset_index(drop=True)

# work out match points, 3 is a win, 1 is a draw, 0 is a loss
match_won = team_matches_long["goals_for"] > team_matches_long["goals_against"]
match_drawn = team_matches_long["goals_for"] == team_matches_long["goals_against"]
team_matches_long["points"] = np.select([match_won, match_drawn], [3, 1], default=0)

print("Rows, columns:", team_matches_long.shape)
print(team_matches_long.columns.tolist())
team_matches_long.head()

# will be used in notebook 03
team_matches_long.to_csv(DATA_FOLDER / "team_matches_long.csv", index=True)

Rows, columns: (760, 9)
['date', 'team', 'opponent', 'venue', 'xg_for', 'xg_against', 'goals_for', 'goals_against', 'points']
                    date                     team           opponent venue  \
0    2025-08-17 15:30:00                  Arsenal  Manchester United  Away   
1    2025-08-23 16:30:00                  Arsenal              Leeds  Home   
2    2025-08-31 15:30:00                  Arsenal          Liverpool  Away   
3    2025-09-13 11:30:00                  Arsenal  Nottingham Forest  Home   
4    2025-09-21 15:30:00                  Arsenal    Manchester City  Home   
..                   ...                      ...                ...   ...   
755  2026-04-25 14:00:00  Wolverhampton Wanderers          Tottenham  Home   
756  2026-05-02 15:00:00  Wolverhampton Wanderers         Sunderland  Home   
757  2026-05-09 15:00:00  Wolverhampton Wanderers           Brighton  Away   
758  2026-05-17 15:00:00  Wolverhampton Wanderers             Fulham  Home   
759  2026-05-24 

In [4]:
league_table = team_matches_long.groupby("team").agg(
    played=("date", "count"),
    points=("points", "sum"),
    goals_for=("goals_for", "sum"),
    goals_against=("goals_against", "sum"),
    xg_for=("xg_for", "sum"),
    xg_against=("xg_against", "sum"),
)

league_table["goal_difference"] = league_table["goals_for"] - league_table["goals_against"]
league_table["xg_difference"] = league_table["xg_for"] - league_table["xg_against"]

# rank each team on points
# rank(ascending=False) means first place goes to the highest value.
league_table["points_rank"] = league_table["points"].rank(ascending=False, method="min").astype(int)
league_table["xg_difference_rank"] = league_table["xg_difference"].rank(ascending=False, method="min").astype(int)

# sort by points like a real league table
league_table = league_table.sort_values("points", ascending=False)
league_table.to_csv(DATA_FOLDER / "league_table.csv", index=True)
league_table.round(1)

,played,points,goals_for,goals_against,xg_for,xg_against,goal_difference,xg_difference,points_rank,xg_difference_rank
team,,,,,,,,,,
Arsenal,38,85,71,27,77.5,33.1,44,44.4,1,1
Manchester City,38,78,77,35,79.4,46.5,42,32.9,2,2
Manchester United,38,71,69,50,72.2,50.3,19,21.9,3,3
Aston Villa,38,65,56,49,56.2,56.7,7,-0.5,4,12
Liverpool,38,60,63,53,67.3,53.9,10,13.4,5,5
Bournemouth,38,57,58,54,66.8,56.8,4,10.1,6,6
Sunderland,38,54,42,48,43.1,60.3,-6,-17.2,7,17
Brentford,38,53,55,52,66.3,59.1,3,7.1,8,7
Brighton,38,53,52,46,58.4,53.8,6,4.7,8,8
